In [14]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, mean_absolute_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')


pd.set_option('display.max_columns', None)
df = pd.read_csv('sales_summary.csv')




# Question 2 — "How much revenue will we make next month?"

**Model:** Time Series Forecasting (Revenue Prediction)

**Features to use:**
- order_date
- revenue
- store_name
- category_name

**Business value:**
- Inventory planning
- staff scheduling
- sales targets.

In [20]:
# Convert to datetime (FIX)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# ─────────────────────────────────────────────
# SHARED DATE FEATURES (used across all models)
# ─────────────────────────────────────────────
df['month'] = df['order_date'].dt.month
df['quarter'] = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.dayofweek
df['year'] = df['order_date'].dt.year

# ══════════════════════════════════════════════
# QUESTION 2 — REVENUE FORECASTING
# ══════════════════════════════════════════════
print("=" * 50)
print("Q2: Revenue Forecasting")
print("=" * 50)

# Feature engineering
q2 = df.copy()

# Numerical → StandardScaler
q2_numerical = ['quantity', 'discount', 'list_price',
                 'month', 'quarter', 'day_of_week', 'year']

# Nominal → OneHotEncoder
q2_nominal   = ['store_name', 'category_name', 'state']

# Ordinal → OrdinalEncoder (order_status: 1=pending, 2=processing, 3=rejected, 4=complete)
q2_ordinal   = ['order_status']
order_status_categories = [[1, 2, 3, 4]]

q2_preprocessor = ColumnTransformer(transformers=[
    ('num',     StandardScaler(),
                q2_numerical),
    ('nom',     OneHotEncoder(handle_unknown='ignore', sparse_output=False),
                q2_nominal),
    ('ord',     OrdinalEncoder(categories=order_status_categories),
                q2_ordinal),
], remainder='drop')

X_q2 = q2[q2_numerical + q2_nominal + q2_ordinal]
y_q2 = q2['revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X_q2, y_q2, test_size=0.2, random_state=42)

q2_pipeline = Pipeline([
    ('preprocessor', q2_preprocessor),
    ('model',        RandomForestRegressor(n_estimators=100, random_state=42))
])

q2_pipeline.fit(X_train, y_train)
y_pred_q2 = q2_pipeline.predict(X_test)

print(f"MAE : ${mean_absolute_error(y_test, y_pred_q2):,.2f}")
print(f"R²  : {r2_score(y_test, y_pred_q2):.4f}")

# Save predictions
q2_results = X_test.copy()
q2_results['actual_revenue']    = y_test.values
q2_results['predicted_revenue'] = y_pred_q2
# q2_results.to_csv(os.path.join(OUT_DIR, 'q2_revenue_forecast.csv'), index=False)
# print("✅ Saved q2_revenue_forecast.csv\n")


# ══════════════════════════════════════════════
# QUESTION 3 — CUSTOMER CHURN
# ══════════════════════════════════════════════
print("=" * 50)
print("Q3: Customer Churn Prediction")
print("=" * 50)

# Aggregate to customer level — one row per customer
snapshot_date = df['order_date'].max()

q3 = df.groupby('customer_name').agg(
    total_orders      = ('order_id',    'nunique'),      # numerical
    total_revenue     = ('revenue',     'sum'),          # numerical
    avg_revenue       = ('revenue',     'mean'),         # numerical
    avg_discount      = ('discount',    'mean'),         # numerical
    avg_quantity      = ('quantity',    'mean'),         # numerical
    days_since_order  = ('order_date',
                         lambda x: (snapshot_date - x.max()).days),  # numerical
    favourite_category= ('category_name', lambda x: x.mode()[0]),    # nominal
    favourite_state   = ('state',         lambda x: x.mode()[0]),    # nominal
).reset_index()

# Target: churned if no order in last 365 days
q3['churned'] = (q3['days_since_order'] > 365).astype(int)
print(f"Churn rate: {q3['churned'].mean():.1%}")

# Numerical → StandardScaler
q3_numerical = ['total_orders', 'total_revenue', 'avg_revenue',
                 'avg_discount', 'avg_quantity', 'days_since_order']

# Nominal → OneHotEncoder
q3_nominal   = ['favourite_category', 'favourite_state']

q3_preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),
            q3_numerical),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            q3_nominal),
], remainder='drop')

X_q3 = q3[q3_numerical + q3_nominal]
y_q3 = q3['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X_q3, y_q3, test_size=0.2, random_state=42, stratify=y_q3)

q3_pipeline = Pipeline([
    ('preprocessor', q3_preprocessor),
    ('model',        RandomForestClassifier(n_estimators=100,
                                            class_weight='balanced',
                                            random_state=42))
])

q3_pipeline.fit(X_train, y_train)
print(classification_report(y_test, q3_pipeline.predict(X_test)))

# Save with churn probability
q3['churn_probability'] = q3_pipeline.predict_proba(X_q3)[:, 1]
q3['churn_risk'] = pd.cut(
    q3['churn_probability'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Low', 'Medium', 'High']
)
# q3.to_csv(os.path.join(OUT_DIR, 'q3_churn_predictions.csv'), index=False)
# print("✅ Saved q3_churn_predictions.csv\n")


# ══════════════════════════════════════════════
# QUESTION 4 — PRODUCT PORTFOLIO OPTIMIZATION
# ══════════════════════════════════════════════
print("=" * 50)
print("Q4: Product Portfolio Optimization")
print("=" * 50)

# Aggregate to product level
q4 = df.groupby(['product_name', 'category_name', 'brand_name']).agg(
    total_revenue  = ('revenue',   'sum'),     # numerical
    units_sold     = ('quantity',  'sum'),     # numerical
    avg_discount   = ('discount',  'mean'),    # numerical
    avg_list_price = ('list_price','mean'),    # numerical
    order_count    = ('order_id',  'nunique'), # numerical
).reset_index()

# Target: is this a HIGH VALUE product?
revenue_threshold = q4['total_revenue'].median()
q4['high_value']  = (q4['total_revenue'] > revenue_threshold).astype(int)

# Numerical → StandardScaler
q4_numerical = ['total_revenue', 'units_sold',
                 'avg_discount', 'avg_list_price', 'order_count']

# Nominal → OneHotEncoder
q4_nominal   = ['category_name', 'brand_name']

q4_preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),
            q4_numerical),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            q4_nominal),
], remainder='drop')

X_q4 = q4[q4_numerical + q4_nominal]
y_q4 = q4['high_value']

X_train, X_test, y_train, y_test = train_test_split(
    X_q4, y_q4, test_size=0.2, random_state=42)

q4_pipeline = Pipeline([
    ('preprocessor', q4_preprocessor),
    ('model',        RandomForestClassifier(n_estimators=100, random_state=42))
])

q4_pipeline.fit(X_train, y_train)
print(classification_report(y_test, q4_pipeline.predict(X_test)))

# Tag each product for dashboard
q4['stock_action'] = np.where(
    (q4['total_revenue'] > revenue_threshold) & (q4['avg_discount'] < 0.1),
    'REORDER NOW',
    np.where(
        (q4['total_revenue'] < revenue_threshold) & (q4['avg_discount'] > 0.15),
        'CONSIDER DROPPING',
        'MONITOR'
    )
)

# q4.to_csv(os.path.join(OUT_DIR, 'q4_product_portfolio.csv'), index=False)
# print("✅ Saved q4_product_portfolio.csv\n")

print("🎉 All 3 models done! Load CSVs from outputs/ into Power BI.")

Q2: Revenue Forecasting
MAE : $6.32
R²  : 0.9996
Q3: Customer Churn Prediction
Churn rate: 81.2%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        54
           1       1.00      1.00      1.00       235

    accuracy                           1.00       289
   macro avg       1.00      1.00      1.00       289
weighted avg       1.00      1.00      1.00       289

Q4: Product Portfolio Optimization
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        32

    accuracy                           1.00        62
   macro avg       1.00      1.00      1.00        62
weighted avg       1.00      1.00      1.00        62

🎉 All 3 models done! Load CSVs from outputs/ into Power BI.
